In [2]:
# Block 1 - Import Libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("All libraries loaded successfully! ✅")

All libraries loaded successfully! ✅


In [8]:
# Block 2 - Load Dataset
df = pd.read_csv('Combined_Funda_data.csv')

# See first 5 rows
print(df.head())

# Check shape
print("\nRows and Columns:", df.shape)

# Check column names
print("\nColumn Names:", df.columns.tolist())

    Aangeboden sinds   Verkoopdatum   Looptijd        Laatste vraagprijs  \
0        19 mei 2022  10 maart 2023   9½ maand    € 485.000 vrij op naam   
1  26 september 2022   22 juni 2023   8½ maand    € 375.000 vrij op naam   
2    25 januari 2023    18 mei 2023   3½ maand    € 279.500 vrij op naam   
3    2 december 2022     9 mei 2023  5 maanden  € 2.330.000 vrij op naam   
4    2 december 2022    15 mei 2023  5 maanden  € 2.270.000 vrij op naam   

                                   Vraagprijs per m²    Status Bijdrage VvE  \
0  € 3.212\r\n    \nDe vraagprijs per m² wordt be...  Verkocht          NaN   
1  € 3.205\r\n    \nDe vraagprijs per m² wordt be...  Verkocht          NaN   
2  € 2.973\r\n    \nDe vraagprijs per m² wordt be...  Verkocht          NaN   
3  € 7.590\r\n    \nLet op! Woning met groot perc...  Verkocht          NaN   
4  € 7.394\r\n    \nLet op! Woning met groot perc...  Verkocht          NaN   

  Soort appartement Soort bouw Bouwperiode  ... Cv-ketel Oppervlakte

In [9]:
# See all column names
for col in df.columns:
    print(col)

Aangeboden sinds
Verkoopdatum
Looptijd
Laatste vraagprijs
Vraagprijs per m²
Status
Bijdrage VvE
Soort appartement
Soort bouw
Bouwperiode
Specifiek
Soort dak
Wonen
Gebouwgebonden buitenruimte
Externe bergruimte
Inhoud
Aantal kamers
Aantal badkamers
Badkamervoorzieningen
Aantal woonlagen
Gelegen op
Voorzieningen
Energielabel
Isolatie
Verwarming
Warm water
Eigendomssituatie
Ligging
Balkon/dakterras
Schuur/berging
Soort parkeergelegenheid
Inschrijving KvK
Jaarlijkse vergadering
Periodieke bijdrage
Reservefonds aanwezig
Onderhoudsplan
Opstalverzekering
Soort woonhuis
Overige inpandige ruimte
Perceel
Cv-ketel
Oppervlakte
Tuin
Achtertuin
Ligging tuin
Soort garage
Capaciteit
Keurmerken
Toegankelijkheid
Zijtuin


In [10]:
# Block 3 - Clean the Data

# Keep only useful columns
df_clean = df[['Laatste vraagprijs', 'Oppervlakte', 'Verkoopdatum',
               'Soort woonhuis', 'Energielabel', 'Aantal kamers']].copy()

# Rename to English
df_clean.columns = ['price', 'area', 'sale_date', 'house_type',
                    'energy_label', 'rooms']

# Clean price column - remove € and text, keep numbers only
df_clean['price'] = df_clean['price'].str.replace('€', '')\
                                     .str.replace('vrij op naam', '')\
                                     .str.replace('kosten koper', '')\
                                     .str.replace('.', '')\
                                     .str.strip()
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')

# Clean area column - remove m²
df_clean['area'] = df_clean['area'].str.replace(' m²', '')\
                                   .str.replace('m²', '')\
                                   .str.replace('.', '')\
                                   .str.strip()
df_clean['area'] = pd.to_numeric(df_clean['area'], errors='coerce')

# Extract year from sale date
df_clean['year'] = pd.to_datetime(df_clean['sale_date'],
                                   format='%d %B %Y', errors='coerce').dt.year

# Drop missing values
df_clean = df_clean.dropna(subset=['price', 'area'])

# Remove outliers
df_clean = df_clean[df_clean['price'] > 10000]
df_clean = df_clean[df_clean['area'] > 10]

print("Data cleaned! ✅")
print("Remaining rows:", len(df_clean))
print(df_clean.head())

Data cleaned! ✅
Remaining rows: 6823
       price    area      sale_date                            house_type  \
0   485000.0   263.0  10 maart 2023  Eengezinswoning, 2-onder-1-kapwoning   
1   375000.0   198.0   22 juni 2023           Eengezinswoning, hoekwoning   
2   279500.0   128.0    18 mei 2023         Eengezinswoning, tussenwoning   
3  2330000.0  2479.0     9 mei 2023             Villa, vrijstaande woning   
4  2270000.0  1796.0    15 mei 2023             Villa, vrijstaande woning   

                          energy_label                     rooms  year  
0     A\r\n        \nWat betekent dit?  5 kamers (4 slaapkamers)   NaN  
1  A+++\r\n        \nWat betekent dit?  5 kamers (4 slaapkamers)   NaN  
2                     Niet beschikbaar  3 kamers (2 slaapkamers)   NaN  
3                     Niet beschikbaar  8 kamers (5 slaapkamers)   NaN  
4                     Niet beschikbaar  8 kamers (5 slaapkamers)   NaN  


In [11]:
# Fix year - Dutch month names
dutch_months = {
    'januari': 'January', 'februari': 'February', 'maart': 'March',
    'april': 'April', 'mei': 'May', 'juni': 'June',
    'juli': 'July', 'augustus': 'August', 'september': 'September',
    'oktober': 'October', 'november': 'November', 'december': 'December'
}

def convert_dutch_date(date_str):
    if pd.isna(date_str):
        return None
    for dutch, english in dutch_months.items():
        date_str = str(date_str).replace(dutch, english)
    try:
        return pd.to_datetime(date_str, format='%d %B %Y')
    except:
        return None

df_clean['sale_date'] = df_clean['sale_date'].apply(convert_dutch_date)
df_clean['year'] = df_clean['sale_date'].dt.year

# Fix energy label - keep only first part
df_clean['energy_label'] = df_clean['energy_label'].str.split('\r\n').str[0].str.strip()

# Fix rooms - extract just the number
df_clean['rooms'] = df_clean['rooms'].str.extract('(\d+)').astype(float)

print("Fixes applied! ✅")
print(df_clean[['price', 'area', 'year', 'energy_label', 'rooms']].head(10))

<>:26: SyntaxWarning: invalid escape sequence '\d'
<>:26: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1299/1660645024.py:26: SyntaxWarning: invalid escape sequence '\d'
  df_clean['rooms'] = df_clean['rooms'].str.extract('(\d+)').astype(float)


Fixes applied! ✅
        price    area  year      energy_label  rooms
0    485000.0   263.0  2023                 A    5.0
1    375000.0   198.0  2023              A+++    5.0
2    279500.0   128.0  2023  Niet beschikbaar    3.0
3   2330000.0  2479.0  2023  Niet beschikbaar    8.0
4   2270000.0  1796.0  2023  Niet beschikbaar    8.0
6    720000.0   105.0  2023  Niet beschikbaar    4.0
7    750000.0   112.0  2023  Niet beschikbaar    4.0
11  2825000.0   886.0  2023  Niet beschikbaar    8.0
16   455000.0   378.0  2022              A+++    5.0
17   425000.0   270.0  2023              A+++    5.0


In [12]:
# Block 4 - SQL Queries

conn = sqlite3.connect('housing.db')
df_clean.to_sql('housing', conn, if_exists='replace', index=False)
print("Data loaded into SQL! ✅")

# Query 1 - Average price by house type
query1 = """
SELECT house_type,
       ROUND(AVG(price), 0) as avg_price,
       COUNT(*) as total_houses
FROM housing
GROUP BY house_type
ORDER BY avg_price DESC
LIMIT 10
"""
result1 = pd.read_sql_query(query1, conn)
print("\nTop 10 House Types by Price:")
print(result1)

# Query 2 - Price trend by year
query2 = """
SELECT year,
       ROUND(AVG(price), 0) as avg_price,
       COUNT(*) as total_sales
FROM housing
WHERE year IS NOT NULL
GROUP BY year
ORDER BY year
"""
result2 = pd.read_sql_query(query2, conn)
print("\nPrice Trend by Year:")
print(result2)

# Query 3 - Average price by energy label
query3 = """
SELECT energy_label,
       ROUND(AVG(price), 0) as avg_price,
       COUNT(*) as total_houses
FROM housing
WHERE energy_label != 'Niet beschikbaar'
GROUP BY energy_label
ORDER BY avg_price DESC
"""
result3 = pd.read_sql_query(query3, conn)
print("\nAverage Price by Energy Label:")
print(result3)

Data loaded into SQL! ✅

Top 10 House Types by Price:
                                          house_type  avg_price  total_houses
0            Villa, vrijstaande woning (waterwoning)  2100000.0             2
1                      Landhuis, 2-onder-1-kapwoning  1800000.0             1
2                                  Villa, hoekwoning  1282500.0             4
3                          Villa, vrijstaande woning  1092320.0           169
4           Villa, 2-onder-1-kapwoning (waterwoning)  1050000.0             1
5                         Villa, 2-onder-1-kapwoning  1038333.0            15
6                      Villa, halfvrijstaande woning   993300.0            10
7  Bungalow, geschakelde 2-onder-1-kapwoning (spl...   940000.0             1
8                       Landhuis, vrijstaande woning   912804.0            28
9        Eengezinswoning, tussenwoning (waterwoning)   879500.0             2

Price Trend by Year:
   year  avg_price  total_sales
0  2022   539464.0           42
1 

In [13]:
# Block 5 - Visualizations

# Chart 1 - Average Price by House Type (Top 10)
fig1 = px.bar(result1,
              x='avg_price',
              y='house_type',
              orientation='h',
              title='Top 10 House Types by Average Price in Netherlands',
              color='avg_price',
              color_continuous_scale='Blues',
              labels={'avg_price': 'Average Price (€)', 'house_type': 'House Type'})
fig1.show()

# Chart 2 - Price Trend by Year
fig2 = px.line(result2,
               x='year',
               y='avg_price',
               title='House Price Trend in Netherlands (2022-2023)',
               markers=True,
               labels={'avg_price': 'Average Price (€)', 'year': 'Year'})
fig2.show()

# Chart 3 - Energy Label vs Price
fig3 = px.bar(result3,
              x='energy_label',
              y='avg_price',
              title='Average House Price by Energy Label',
              color='avg_price',
              color_continuous_scale='Greens',
              labels={'avg_price': 'Average Price (€)',
                      'energy_label': 'Energy Label'})
fig3.show()

# Chart 4 - Area vs Price Scatter
fig4 = px.scatter(df_clean[df_clean['price'] < 2000000],
                  x='area',
                  y='price',
                  color='energy_label',
                  title='House Area vs Price',
                  labels={'area': 'Area (sqm)', 'price': 'Price (€)'},
                  opacity=0.6)
fig4.show()

print("All charts created! ✅")

All charts created! ✅


In [14]:
# Block 6 - Price Prediction Model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Prepare data - use area and rooms to predict price
df_model = df_clean[['area', 'rooms', 'price']].dropna()

X = df_model[['area', 'rooms']]
y = df_model['price']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Test the model
y_pred = model.predict(X_test)

# Results
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Model Trained! ✅")
print(f"R² Score: {r2:.2f}")
print(f"RMSE: €{rmse:,.0f}")

Model Trained! ✅
R² Score: 0.21
RMSE: €199,992


In [15]:
# Block 7 - Visualize Predictions

# Actual vs Predicted chart
fig5 = px.scatter(
    x=y_test,
    y=y_pred,
    title='Actual vs Predicted House Prices',
    labels={'x': 'Actual Price (€)', 'y': 'Predicted Price (€)'},
    opacity=0.5
)
fig5.add_shape(
    type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash')
)
fig5.show()

# Try predicting a house yourself!
print("\n--- Price Predictor ---")
test_house = pd.DataFrame({'area': [100], 'rooms': [4]})
predicted_price = model.predict(test_house)[0]
print(f"A 100sqm house with 4 rooms → Predicted Price: €{predicted_price:,.0f}")

test_house2 = pd.DataFrame({'area': [200], 'rooms': [6]})
predicted_price2 = model.predict(test_house2)[0]
print(f"A 200sqm house with 6 rooms → Predicted Price: €{predicted_price2:,.0f}")


--- Price Predictor ---
A 100sqm house with 4 rooms → Predicted Price: €368,202
A 200sqm house with 6 rooms → Predicted Price: €518,806
